# Finite-memory information MDP + Q-learning (mapping)

Interactive smoke notebook for the `finite_memory` package (sibling of `belief_quantized`; shared costs/quantizers in `classes`).

- Known-pose active mapping with a tiny `LandmarkMap`
- Continuous Y/U quantized to finite alphabets
- Approximate information MDP with fixed prior $b^*$ and memory $N$
- Tabular finite-memory Q-learning using shared `classes.costs`


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('USE_CUPY', 'false')

import numpy as np
import matplotlib.pyplot as plt

from src.belief_quantized.belief_mdp_n import BeliefMDP_n_Mapping
from src.classes.mapping import LandmarkMap
from src.classes.model import RangeBearingSensor, SingleIntegratorModel
from src.classes.costs import shannon_entropy_rows
from src.finite_memory import (
    ApproximateInformationMDP,
    FiniteMemoryQLearning,
    MappingRollout,
)

def shannon_entropy(b):
    return float(shannon_entropy_rows(np.atleast_2d(np.asarray(b, dtype=float)))[0])


def make_tiny_mapping_mdp(
    *,
    pose_n=2,
    map_n=2,
    obs_n=2,
    action_n=2,
    num_landmarks=1,
):
    landmark_map = LandmarkMap(
        x_min=0.0, x_max=10.0, y_min=0.0, y_max=10.0,
        n=map_n, num_landmarks=num_landmarks,
    )
    motion_model = SingleIntegratorModel(i_x=5.0, i_y=5.0, dt=1.0, max_v=4.0)
    epsilon, r_max = 0.1, 4.0
    sensor = RangeBearingSensor(
        r_max=r_max, epsilon=epsilon, sigma_r=0.5, sigma_phi=0.3,
        r0=epsilon + 0.1 * (r_max - epsilon),
        r1=epsilon + 0.9 * (r_max - epsilon),
    )
    cov_y = np.diag(np.tile([0.5**2, 0.3**2], num_landmarks))
    return BeliefMDP_n_Mapping(
        n=pose_n,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=[],
        _map=landmark_map,
        sigma_w=0.1,
        cov_y=cov_y,
        exploration_type='information gain',
        obs_n=obs_n,
        action_n=action_n,
    )

print('CWD:', Path.cwd())


In [ ]:
# Tiny quantization / memory config
pose_n = 2
map_n = 2
obs_n = 2
action_n = 2
memory_N = 0  # try 1 once |D^N| is comfortable
lam = 10.0
beta = 0.95

mdp = make_tiny_mapping_mdp(
    pose_n=pose_n,
    map_n=map_n,
    obs_n=obs_n,
    action_n=action_n,
    num_landmarks=1,
)
approx = ApproximateInformationMDP(
    mdp, memory_N=memory_N, lam=lam, exploration_type='information gain'
)
rollout = MappingRollout(approx, seed=0)

n_y = int(mdp.Y_n.shape[0])
n_u = int(mdp.AQ.n_u)
print(f'|Y_n|={n_y}, |U_n|={n_u}, |M|={mdp.len_M}')
print(f'|D^N|={approx.n_states} (N={memory_N})')


In [ ]:
# Inspect Psi(b*, h) and stage cost on a reset window
h, info = rollout.reset(true_map=0)
b = info['belief']
print('h index:', h)
print('belief:', np.round(b, 4))
print('entropy:', shannon_entropy(b))
print('stage cost at u=0:', approx.stage_cost(h, 0, rollout.pose_hist))


In [ ]:
# Train finite-memory Q-learning
ql = FiniteMemoryQLearning(
    approx, rollout, beta=beta, epsilon=0.3, exploration='epsilon_greedy', seed=1
)

n_episodes = 40
horizon = 15
true_maps = list(range(mdp.len_M))
Q = ql.train(n_episodes=n_episodes, horizon=horizon, true_maps=true_maps, verbose=True)

plt.figure(figsize=(6, 3))
plt.plot(ql.episode_returns)
plt.xlabel('episode')
plt.ylabel('undiscounted return')
plt.title('Q-learning episode returns')
plt.tight_layout()
plt.show()


In [ ]:
# Greedy vs random entropy trajectories on one true map
true_map = 0
horizon_eval = 20

greedy = ql.rollout_greedy(horizon_eval, true_map=true_map, epsilon=0.0)

# random baseline using the same rollout API
rng = np.random.default_rng(123)
h, info = rollout.reset(true_map=true_map)
rand_ent = [shannon_entropy(info['belief'])]
rand_cost = []
for _ in range(horizon_eval):
    u = int(rng.integers(0, approx.n_actions))
    _, c, h, info = rollout.step(u)
    rand_cost.append(c)
    rand_ent.append(shannon_entropy(info['belief']))

plt.figure(figsize=(6, 3))
plt.plot(greedy['entropies'], label='greedy Q')
plt.plot(rand_ent, label='random')
plt.xlabel('t')
plt.ylabel('map belief entropy')
plt.legend()
plt.title(f'Entropy rollout (true map={true_map})')
plt.tight_layout()
plt.show()

print('greedy total cost:', greedy['total_cost'])
print('random total cost:', float(np.sum(rand_cost)))
